## Section 1 — Import Libraries

In [1]:
# ============================================================
# SECTION 1 : Imports
# ============================================================

# ---------- System ----------
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

# ---------- Data ----------
import numpy as np
import pandas as pd

# ---------- Model ----------
import joblib

# ---------- PDF ----------
import fitz  # PyMuPDF

# ---------- NLP ----------
import nltk
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- Report ----------
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import (
    SimpleDocTemplate,
    Table,
    TableStyle,
    Paragraph
)

# ---------- Display ----------
from IPython.display import display

print("=" * 60)
print("FAIRHIRE")
print("Candidate Prediction Notebook")
print("=" * 60)

print("\nLibraries Loaded Successfully")

# ============================================================
# SECTION 1 : Libraries
# ============================================================

import sys

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

UTILS_DIR = PROJECT_ROOT / "utils"

sys.path.append(str(UTILS_DIR))

print("✓ Utilities Loaded")

# try:
#     import importlib

#     prediction_utils = importlib.import_module("prediction_utils")
#     globals().update(
#         {
#             name: getattr(prediction_utils, name)
#             for name in dir(prediction_utils)
#             if not name.startswith("_")
#         }
#     )
#     print("✓ prediction_utils imported")
# except Exception as e:
#     print(f"prediction_utils not found in utils: {e}")
#     # Fallback: provide minimal PDF extraction utility using PyMuPDF (fitz)
#     def extract_text_from_pdf(pdf_path):
#         """
#         Fallback PDF text extractor using fitz (PyMuPDF).
#         Returns empty string on failure.
#         """
#         try:
#             doc = fitz.open(str(pdf_path))
#             text = []
#             for page in doc:
#                 page_text = page.get_text()
#                 if page_text:
#                     text.append(page_text)
#             return "\n".join(text)
#         except Exception as exc:
#             print(f"Fallback extract_text_from_pdf failed for {pdf_path}: {exc}")
#             return ""

FAIRHIRE
Candidate Prediction Notebook

Libraries Loaded Successfully
✓ Utilities Loaded


## SECTION 2 : Configuration & Project Paths

In [2]:
# ============================================================
# SECTION 2 : Configuration & Project Paths
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Project Root
# ------------------------------------------------------------

PROJECT_ROOT = Path("..")

# ------------------------------------------------------------
# Folders
# ------------------------------------------------------------

MODELS_DIR = PROJECT_ROOT / "models"

DATASET_DIR = PROJECT_ROOT / "dataset"

RESUME_DIR = DATASET_DIR / "resumes"

JOB_DIR = DATASET_DIR / "job_descriptions"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

# ------------------------------------------------------------
# Resume Folder
# ------------------------------------------------------------

RESUME_CATEGORY = "Cloud DevOps and SRE"

resume_folder = RESUME_DIR / RESUME_CATEGORY

# ------------------------------------------------------------
# Job Description
# ------------------------------------------------------------

JOB_FILE = JOB_DIR / "cloud_devops_sre.txt"

# ------------------------------------------------------------
# Create Output Folder
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Display Configuration
# ------------------------------------------------------------

print("=" * 60)
print("Project Configuration")
print("=" * 60)

print(f"Project Root      : {PROJECT_ROOT.resolve()}")
print(f"Models Folder     : {MODELS_DIR.resolve()}")
print(f"Resume Folder     : {resume_folder.resolve()}")
print(f"Job Description   : {JOB_FILE.resolve()}")
print(f"Output Folder     : {OUTPUT_DIR.resolve()}")

Project Configuration
Project Root      : D:\Semesters\6th_Sem + Minor\Minor Project\FairHire
Models Folder     : D:\Semesters\6th_Sem + Minor\Minor Project\FairHire\models
Resume Folder     : D:\Semesters\6th_Sem + Minor\Minor Project\FairHire\dataset\resumes\Cloud DevOps and SRE
Job Description   : D:\Semesters\6th_Sem + Minor\Minor Project\FairHire\dataset\job_descriptions\cloud_devops_sre.txt
Output Folder     : D:\Semesters\6th_Sem + Minor\Minor Project\FairHire\outputs


SECTION 2.1 : Collect Resume Files

In [3]:
resume_folder = PROJECT_ROOT / "real_test_data" / "resumes" / "Cloud, DevOps and SRE"

SECTION 2.2 : Collect Resume PDFs

In [4]:
resume_files = sorted(resume_folder.glob("*.pdf"))

print(f"Total Resume PDFs : {len(resume_files)}")
print()

for i, pdf in enumerate(resume_files, start=1):
    print(f"{i:02d}. {pdf.name}")

Total Resume PDFs : 8

01. AWS-Engineer-Resume.pdf
02. Azure-Engineer-Resume.pdf
03. Cloud-Engineer-Resume.pdf
04. DevOps-Engineer-Resume.pdf
05. DevSecOps-Engineer-Resume.pdf
06. Infrastructure-Engineer-Resume.pdf
07. Platform-Engineer-Resume.pdf
08. Site-Reliability-Engineer-Resume.pdf


In [5]:
print(PROJECT_ROOT.resolve())
print(resume_folder.resolve())
print(resume_folder.exists())

D:\Semesters\6th_Sem + Minor\Minor Project\FairHire
D:\Semesters\6th_Sem + Minor\Minor Project\FairHire\real_test_data\resumes\Cloud, DevOps and SRE
True


## SECTION 3 : Load All Models

In [6]:
# ============================================================
# Debug : Show Model Files
# ============================================================

for file in MODELS_DIR.rglob("*"):
    print(file.relative_to(MODELS_DIR))

best_model.pkl
fairhire_model.pkl
feature_names.pkl
job_embeddings.pkl
linear_svm.pkl
logistic_regression.pkl
project_summary.pkl
random_forest.pkl
resume_embeddings.pkl
scaler.pkl
skill_dictionary.pkl
tfidf_vectorizer.pkl
top_features.pkl
trained_models
vectorizer.pkl
xgboost.pkl
X_test_scaled.pkl
X_train_scaled.pkl
y_test.pkl
y_train.pkl
trained_models\dummy_classifier.pkl
trained_models\linear_svm.pkl
trained_models\logistic_regression.pkl
trained_models\multinomial_naive_bayes.pkl
trained_models\random_forest.pkl
trained_models\xgboost.pkl


In [7]:
# ============================================================
# SECTION 3 : Load Models
# ============================================================

print("=" * 60)
print("Loading FairHire Models")
print("=" * 60)

# ------------------------------------------------------------
# Best XGBoost Model
# ------------------------------------------------------------

best_model = joblib.load(
    MODELS_DIR / "best_model.pkl"
)

print("✓ Best Model Loaded")


# ------------------------------------------------------------
# TF-IDF Vectorizer
# ------------------------------------------------------------

vectorizer = joblib.load(
    MODELS_DIR / "tfidf_vectorizer.pkl"
)

print("✓ TF-IDF Vectorizer Loaded")


# ------------------------------------------------------------
# Standard Scaler
# ------------------------------------------------------------

scaler = joblib.load(
    MODELS_DIR / "scaler.pkl"
)

print("✓ Scaler Loaded")


# ------------------------------------------------------------
# Skill Dictionary
# ------------------------------------------------------------

skill_dictionary = joblib.load(
    MODELS_DIR / "skill_dictionary.pkl"
)

print("✓ Skill Dictionary Loaded")


# ------------------------------------------------------------
# Sentence Transformer
# ------------------------------------------------------------

sentence_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✓ Sentence Transformer Loaded")

print("=" * 60)

Loading FairHire Models


✓ Best Model Loaded
✓ TF-IDF Vectorizer Loaded
✓ Scaler Loaded
✓ Skill Dictionary Loaded


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Sentence Transformer Loaded


In [8]:
# ============================================================
# SECTION 3.1 : Verify Models
# ============================================================

print("Model Features :", best_model.n_features_in_)

print("Vocabulary Size :", len(vectorizer.get_feature_names_out()))

print("Total Skills :", len(skill_dictionary))

print()

print(type(best_model))

print(type(vectorizer))

print(type(scaler))

print(type(skill_dictionary))

Model Features : 156
Vocabulary Size : 156
Total Skills : 1882

<class 'xgboost.sklearn.XGBClassifier'>
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'sklearn.preprocessing._data.StandardScaler'>
<class 'set'>


## SECTION 4 : Helper Functions

In [9]:
# ============================================================
# SECTION 4 : Load Job Description
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Job Description Path
# ------------------------------------------------------------

job_description_file = (
    PROJECT_ROOT
    / "real_test_data"
    / "job_descriptions"
    / "cloud_devops_sre.txt"
)

print(job_description_file)
print(job_description_file.exists())

..\real_test_data\job_descriptions\cloud_devops_sre.txt
True


In [10]:
# ============================================================
# SECTION 4.1 : Read Job Description
# ============================================================

with open(
    job_description_file,
    "r",
    encoding="utf-8"
) as file:

    job_description = file.read()

print("Job Description Loaded Successfully")
print()

print(job_description[:1000])

Job Description Loaded Successfully

Job Title: Cloud Engineer

Company Overview

We are looking for a Cloud Engineer to design, implement, maintain, and optimize cloud infrastructure across AWS, Azure, and Google Cloud Platform. The ideal candidate should possess strong knowledge of cloud services, Infrastructure as Code (IaC), containerization, automation, monitoring, and cloud security.

Key Responsibilities

• Design and deploy scalable cloud infrastructure.
• Build highly available and fault-tolerant cloud architectures.
• Manage AWS, Azure, or GCP cloud environments.
• Develop Infrastructure as Code using Terraform or CloudFormation.
• Configure Kubernetes clusters and Docker containers.
• Automate deployments using CI/CD pipelines.
• Monitor cloud infrastructure using CloudWatch, Datadog, Prometheus, or Grafana.
• Implement IAM policies and cloud security best practices.
• Optimize cloud costs and resource utilization.
• Collaborate with software developers and DevOps teams.

Re

In [11]:
# ============================================================
# SECTION 4.2 : Load Job Description Function
# ============================================================

def load_job_description(file_path):
    """
    Read Job Description text file.
    """

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        return file.read()

In [12]:
# ============================================================
# SECTION 4.3 : Verification
# ============================================================

job_description = load_job_description(job_description_file)

print("Characters :", len(job_description))
print("Words      :", len(job_description.split()))

Characters : 1529
Words      : 202


## Section 5 — Resume Extraction

In [13]:
# ============================================================
# SECTION 5 : Resume Extraction
# ============================================================

import pdfplumber
from PyPDF2 import PdfReader


# ------------------------------------------------------------
# Extract Resume Text
# ------------------------------------------------------------

def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF resume.

    Priority:
        1. pdfplumber
        2. PyPDF2 (fallback)

    Returns
    -------
    str
        Extracted resume text.
    """

    text = ""

    # --------------------------------------------------------
    # Try pdfplumber first
    # --------------------------------------------------------

    try:

        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:

                    text += page_text + "\n"

    except Exception:

        text = ""

    # --------------------------------------------------------
    # Fallback using PyPDF2
    # --------------------------------------------------------

    if len(text.strip()) == 0:

        try:

            reader = PdfReader(pdf_path)

            for page in reader.pages:

                page_text = page.extract_text()

                if page_text:

                    text += page_text + "\n"

        except Exception as e:

            print(f"Error reading {pdf_path.name}")
            print(e)

            return ""

    # --------------------------------------------------------
    # Basic Cleaning
    # --------------------------------------------------------

    text = text.replace("\xa0", " ")

    text = text.replace("\t", " ")

    text = text.replace("\r", " ")

    text = "\n".join(
        line.strip()
        for line in text.splitlines()
        if line.strip()
    )

    return text


# ------------------------------------------------------------
# Extract All Resume Texts
# ------------------------------------------------------------

def load_all_resumes(resume_files):
    """
    Extract text from every resume.

    Returns
    -------
    list
    """

    resume_texts = []

    for pdf in resume_files:

        resume_text = extract_text_from_pdf(pdf)

        resume_texts.append(resume_text)

    return resume_texts


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 70)
print("Testing Resume Extraction")
print("=" * 70)

sample_resume = extract_text_from_pdf(resume_files[0])

print("Candidate :", resume_files[0].stem)
print()

print("Characters :", len(sample_resume))
print("Words      :", len(sample_resume.split()))

print()
print("=" * 70)
print(sample_resume[:1200])
print("=" * 70)

Testing Resume Extraction
Candidate : AWS-Engineer-Resume

Characters : 7487
Words      : 1040

Anh Nguyen
Senior AWS Engineer Sydney, Australia • awseng@gmail.com • +61 2 5550 7184
PROFILE SUMMARY
Senior AWS Engineer with 8 years of experience running AWS production environments at consumer SaaS scale across
developer collaboration, productivity SaaS, and ITSM platforms, specializing in Well-Architected reviews, EKS on Fargate,
and Terraform-driven landing zones.
Hands-on coverage across compute (EKS on Fargate), IaC (Terraform), CI/CD (GitHub Actions), observability (CloudWatch
with Datadog), and serverless (Lambda with Step Functions), with networking grounded in hub-and-spoke Transit Gateway
with PrivateLink and certified on AWS Solutions Architect Professional.
Deep expertise in Well-Architected reviews across all six pillars, multi-account landing zones with Control Tower, event-
driven serverless with EventBridge and Step Functions, and FinOps tagging and Savings Plans optimizat

## SECTION 6 : Text Preprocessing

In [14]:
# ============================================================
# SECTION 6 : Text Preprocessing
# ============================================================

import re
import string

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# ------------------------------------------------------------
# Download NLTK Resources
# ------------------------------------------------------------

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ------------------------------------------------------------
# Initialize Objects
# ------------------------------------------------------------

STOP_WORDS = set(stopwords.words("english"))

LEMMATIZER = WordNetLemmatizer()

# ------------------------------------------------------------
# Text Cleaning Function
# ------------------------------------------------------------

def preprocess_text(text):
    """
    Clean Resume / Job Description text.
    """

    if text is None:
        return ""

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+", " ", text)

    # Remove Email
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove Phone Numbers
    text = re.sub(r"\+?\d[\d\-\(\)\s]{7,}", " ", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove Numbers
    text = re.sub(r"\d+", " ", text)

    # Remove Extra Spaces
    text = re.sub(r"\s+", " ", text)

    words = []

    for word in text.split():

        if word not in STOP_WORDS:

            word = LEMMATIZER.lemmatize(word)

            words.append(word)

    return " ".join(words)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

clean_resume = preprocess_text(sample_resume)

clean_job = preprocess_text(job_description)

print("="*70)
print("Resume Words :", len(clean_resume.split()))
print("Job Words    :", len(clean_job.split()))
print("="*70)

print(clean_resume[:800])

Resume Words : 791
Job Words    : 172
anh nguyen senior aws engineer sydney australia • • profile summary senior aws engineer year experience running aws production environment consumer saas scale across developer collaboration productivity saas itsm platform specializing wellarchitected review eks fargate terraformdriven landing zone handson coverage across compute eks fargate iac terraform cicd github action observability cloudwatch datadog serverless lambda step function networking grounded hubandspoke transit gateway privatelink certified aws solution architect professional deep expertise wellarchitected review across six pillar multiaccount landing zone control tower event driven serverless eventbridge step function finops tagging saving plan optimization applying methodology gitopsstyle terraform reusable module remote s


## SECTION 7 : Experience Extraction

In [15]:
# ============================================================
# SECTION 7 : Experience Extraction
# ============================================================

import re


# ------------------------------------------------------------
# Experience Extraction
# ------------------------------------------------------------

def extract_years_of_experience(text):
    """
    Extract total years of experience.

    Returns integer years.
    """

    text = text.lower()

    patterns = [

        r'(\d+)\+?\s*years',

        r'(\d+)\+?\s*year',

        r'(\d+)\s*yrs',

        r'(\d+)\s*yr'

    ]

    years = []

    for pattern in patterns:

        matches = re.findall(pattern, text)

        years.extend(matches)

    years = [int(x) for x in years]

    if len(years) == 0:

        return 0

    return max(years)


# ------------------------------------------------------------
# Experience Matching
# ------------------------------------------------------------

def experience_match_score(
        resume_text,
        job_text
):

    resume_exp = extract_years_of_experience(
        resume_text
    )

    job_exp = extract_years_of_experience(
        job_text
    )

    if job_exp == 0:

        score = 100

    else:

        score = min(
            (resume_exp / job_exp) * 100,
            100
        )

    return resume_exp, job_exp, round(score,2)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

resume_exp, job_exp, exp_score = experience_match_score(
    sample_resume,
    job_description
)

print("="*70)

print("Resume Experience :", resume_exp)

print("Job Experience    :", job_exp)

print("Experience Score  :", exp_score)

print("="*70)

Resume Experience : 8
Job Experience    : 3
Experience Score  : 100


## SECTION 8 : Skill Extraction

In [16]:
# ============================================================
# SECTION 8 : Skill Extraction
# ============================================================

import re

import joblib


# ------------------------------------------------------------
# Load Skill Dictionary
# ------------------------------------------------------------

skill_dictionary = joblib.load(
    MODELS_DIR / "skill_dictionary.pkl"
)


# ------------------------------------------------------------
# Extract Skills
# ------------------------------------------------------------

def extract_skills(text):
    """
    Extract skills from text.
    """

    text = text.lower()

    extracted = set()

    for skill in skill_dictionary:

        pattern = r"\b" + re.escape(skill.lower()) + r"\b"

        if re.search(pattern, text):

            extracted.add(skill.lower())

    return sorted(extracted)


# ------------------------------------------------------------
# Skill Comparison
# ------------------------------------------------------------

def compare_skills(
        resume_text,
        job_text
):

    resume_skills = set(
        extract_skills(resume_text)
    )

    job_skills = set(
        extract_skills(job_text)
    )

    matched = sorted(
        resume_skills & job_skills
    )

    missing = sorted(
        job_skills - resume_skills
    )

    return (

        resume_skills,

        job_skills,

        matched,

        missing

    )


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

resume_skills, job_skills, matched_skills, missing_skills = compare_skills(

    clean_resume,

    clean_job

)

print("="*70)

print("Resume Skills :", len(resume_skills))

print("Job Skills    :", len(job_skills))

print("Matched       :", len(matched_skills))

print("Missing       :", len(missing_skills))

print()

print("Matched Skills")

print(matched_skills)

print()

print("Missing Skills")

print(missing_skills)

print("="*70)

Resume Skills : 34
Job Skills    : 31
Matched       : 15
Missing       : 16

Matched Skills
['automation', 'aws', 'cloud', 'cloudformation', 'cloudwatch', 'devops', 'github', 'grafana', 'iam', 'jenkins', 'networking', 'python', 'security', 'terraform', 'vpc']

Missing Skills
['agile', 'ansible', 'azure', 'bash', 'cloud security', 'communication', 'docker', 'docker kubernetes', 'gcp', 'git', 'git github', 'helm', 'kubernetes', 'linux', 'monitoring', 'prometheus']


## SECTION 9 : Keyword Coverage

In [17]:
# ============================================================
# SECTION 9 : Keyword Coverage
# ============================================================

# ------------------------------------------------------------
# Keyword Coverage
# ------------------------------------------------------------

def keyword_coverage(
    resume_text,
    job_description
):
    """
    Calculate keyword coverage between
    resume and job description.
    """

    _, _, matched, missing = compare_skills(
        resume_text,
        job_description
    )

    total = len(matched) + len(missing)

    if total == 0:
        return (
            0.0,
            matched,
            missing
        )

    coverage = (
        len(matched) / total
    ) * 100

    return (

        round(coverage, 2),

        matched,

        missing

    )


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

keyword_score, matched_skills, missing_skills = keyword_coverage(

    clean_resume,

    clean_job

)

print("="*70)

print("Keyword Coverage :", keyword_score)

print()

print("Matched :", len(matched_skills))

print("Missing :", len(missing_skills))

print("="*70)

Keyword Coverage : 48.39

Matched : 15
Missing : 16


In [18]:
# ============================================================
# SECTION 10 : Semantic Similarity
# ============================================================

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# Load Sentence Transformer
# ------------------------------------------------------------

sentence_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ------------------------------------------------------------
# Semantic Similarity
# ------------------------------------------------------------

def semantic_similarity(
    resume_text,
    job_description
):

    resume_embedding = sentence_model.encode(
        resume_text,
        convert_to_numpy=True
    )

    job_embedding = sentence_model.encode(
        job_description,
        convert_to_numpy=True
    )

    similarity = cosine_similarity(

        [resume_embedding],

        [job_embedding]

    )[0][0]

    return round(float(similarity), 4)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

semantic_score = semantic_similarity(

    clean_resume,

    clean_job

)

print("="*70)

print("Semantic Similarity :", semantic_score)

print("="*70)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic Similarity : 0.8113


In [19]:
# ============================================================
# SECTION 11 : Candidate Score
# ============================================================


# ------------------------------------------------------------
# Candidate Score
# ------------------------------------------------------------

def calculate_candidate_score(
        selection_probability, 

        keyword_coverage,

        semantic_similarity,

        experience_score,

        matched_count,

        missing_count

):

    keyword_score = float(keyword_coverage)

    semantic_score = float(semantic_similarity) * 100

    experience_score = float(experience_score)

    total_skills = matched_count + missing_count

    if total_skills == 0:

        skill_score = 0

    else:

        skill_score = (

            matched_count /

            total_skills

        ) * 100


    candidate_score = (
        0.05 * float(selection_probability) +  
        0.30 * keyword_score +                  
        0.30 * semantic_score +                 
        0.15 * experience_score +               
        0.20 * skill_score 
    )

    return round(candidate_score,2)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

candidate_score = calculate_candidate_score(
    globals().get("selection_probability", 0),
    keyword_score,
    semantic_score,
    exp_score,
    len(matched_skills),
    len(missing_skills)
)

print("="*70)

print("Candidate Score :", candidate_score)

print("="*70)

Candidate Score : 63.53


In [20]:
# ============================================================
# SECTION 12 : Candidate Prediction
# ============================================================

import inspect
import numpy as np


# ------------------------------------------------------------
# Candidate Prediction
# ------------------------------------------------------------

def predict_candidate(

        resume_text,

        job_description

):

    # ------------------------------------------
    # Clean Text
    # ------------------------------------------

    clean_resume = preprocess_text(

        resume_text

    )

    clean_job = preprocess_text(

        job_description

    )


    # ------------------------------------------
    # ML Prediction
    # ------------------------------------------

    combined_text = clean_resume + " " + clean_job

    tfidf_vector = vectorizer.transform(

        [combined_text]

    )

    probability = best_model.predict_proba(

        tfidf_vector

    )[0][1]

    probability = round(

        probability * 100,

        2

    )


    # ------------------------------------------
    # Keyword Coverage
    # ------------------------------------------

    keyword_score, matched, missing = keyword_coverage(

        clean_resume,

        clean_job

    )


    # ------------------------------------------
    # Semantic Similarity
    # ------------------------------------------

    semantic_score = semantic_similarity(

        clean_resume,

        clean_job

    )


    # ------------------------------------------
    # Experience
    # ------------------------------------------

    resume_exp, job_exp, experience_score = experience_match_score(

        resume_text,

        job_description

    )


    # ------------------------------------------
    # Candidate Score
    # ------------------------------------------
    candidate_score_args = [

        keyword_score,

        semantic_score,

        experience_score,

        len(matched),

        len(missing),

        probability

    ]

    candidate_score_params = inspect.signature(
        calculate_candidate_score
    ).parameters

    candidate_score = calculate_candidate_score(

        *candidate_score_args[:len(candidate_score_params)]

    )


    # ------------------------------------------
    # Final Decision
    # ------------------------------------------

    decision = (

        "SELECTED"

        if candidate_score >= 75

        else

        "REJECTED"

    )


    # ------------------------------------------
    # Return
    # ------------------------------------------

    return {

        "Decision": decision,

        "Probability": probability,

        "Keyword Coverage": keyword_score,

        "Semantic Similarity": semantic_score,

        "Experience Match": f"{resume_exp}/{job_exp}",

        "Experience Score": experience_score,

        "Candidate Score": candidate_score,

        "Matched Skills": matched,

        "Missing Skills": missing,

        "Matched Count": len(matched),

        "Missing Count": len(missing)

    }


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

result = predict_candidate(

    sample_resume,

    job_description

)

print("="*70)

for key, value in result.items():

    print(f"{key:22}: {value}")

print("="*70)

Decision              : SELECTED
Probability           : 16.059999465942383
Keyword Coverage      : 48.39
Semantic Similarity   : 0.8113
Experience Match      : 8/3
Experience Score      : 100
Candidate Score       : 3014.889892578125
Matched Skills        : ['automation', 'aws', 'cloud', 'cloudformation', 'cloudwatch', 'devops', 'github', 'grafana', 'iam', 'jenkins', 'networking', 'python', 'security', 'terraform', 'vpc']
Missing Skills        : ['agile', 'ansible', 'azure', 'bash', 'cloud security', 'communication', 'docker', 'docker kubernetes', 'gcp', 'git', 'git github', 'helm', 'kubernetes', 'linux', 'monitoring', 'prometheus']
Matched Count         : 15
Missing Count         : 16


In [21]:
# ============================================================
# SECTION 12 : Candidate Prediction
# ============================================================

import numpy as np


# ------------------------------------------------------------
# Candidate Prediction
# ------------------------------------------------------------

def predict_candidate(

        resume_text,

        job_description

):

    # ------------------------------------------
    # Clean Text
    # ------------------------------------------

    clean_resume = preprocess_text(

        resume_text

    )

    clean_job = preprocess_text(

        job_description

    )


    # ------------------------------------------
    # ML Prediction
    # ------------------------------------------

    combined_text = clean_resume + " " + clean_job

    tfidf_vector = vectorizer.transform(

        [combined_text]

    )

    probability = best_model.predict_proba(

        tfidf_vector

    )[0][1]

    probability = round(

        probability * 100,

    )


    # ------------------------------------------
    # Keyword Coverage
    # ------------------------------------------

    keyword_score, matched, missing = keyword_coverage(

        clean_resume,

        clean_job

    )


    # ------------------------------------------
    # Semantic Similarity
    # ------------------------------------------

    semantic_score = semantic_similarity(

        clean_resume,

        clean_job

    )


    # ------------------------------------------
    # Experience
    # ------------------------------------------

    resume_exp, job_exp, experience_score = experience_match_score(

        resume_text,

        job_description

    )


    # ------------------------------------------
    # Candidate Score
    # ------------------------------------------

    candidate_score = calculate_candidate_score(
        probability,

        keyword_score,

        semantic_score,

        experience_score,

        len(matched),

        len(missing)

    )


    # ------------------------------------------
    # Final Decision
    # ------------------------------------------

    decision = (

        "SELECTED"

        if candidate_score >= 75

        else

        "REJECTED"

    )


    # ------------------------------------------
    # Return
    # ------------------------------------------

    return {

        "Decision": decision,

        "Probability": probability,

        "Keyword Coverage": keyword_score,

        "Semantic Similarity": semantic_score,

        "Experience Match": f"{resume_exp}/{job_exp}",

        "Experience Score": experience_score,

        "Candidate Score": candidate_score,

        "Matched Skills": matched,

        "Missing Skills": missing,

        "Matched Count": len(matched),

        "Missing Count": len(missing)

    }


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

result = predict_candidate(

    sample_resume,

    job_description

)

print("="*70)

for key, value in result.items():

    print(f"{key:22}: {value}")

print("="*70)

Decision              : REJECTED
Probability           : 16
Keyword Coverage      : 48.39
Semantic Similarity   : 0.8113
Experience Match      : 8/3
Experience Score      : 100
Candidate Score       : 64.33
Matched Skills        : ['automation', 'aws', 'cloud', 'cloudformation', 'cloudwatch', 'devops', 'github', 'grafana', 'iam', 'jenkins', 'networking', 'python', 'security', 'terraform', 'vpc']
Missing Skills        : ['agile', 'ansible', 'azure', 'bash', 'cloud security', 'communication', 'docker', 'docker kubernetes', 'gcp', 'git', 'git github', 'helm', 'kubernetes', 'linux', 'monitoring', 'prometheus']
Matched Count         : 15
Missing Count         : 16


## SECTION 13 : Predict All Candidates

In [22]:
results_list = []

for idx, pdf_file in enumerate(resume_files, start=1):

    print("=" * 70)
    print(f"Processing Resume {idx}/{len(resume_files)}")
    print(pdf_file.stem)
    print("=" * 70)

    resume_text = extract_text_from_pdf(pdf_file)

    def predict_candidate(resume_text, job_description):
        clean_resume = preprocess_text(resume_text)
        clean_job = preprocess_text(job_description)

        combined_text = clean_resume + " " + clean_job

        tfidf_vector = vectorizer.transform([combined_text])
        probability = best_model.predict_proba(tfidf_vector)[0][1]
        probability = round(probability * 100, 2)

        keyword_score, matched, missing = keyword_coverage(clean_resume, clean_job)
        semantic_score = semantic_similarity(clean_resume, clean_job)

        resume_exp, job_exp, experience_score = experience_match_score(
            resume_text,
            job_description
        )

        candidate_score = calculate_candidate_score(
            probability,
            keyword_score,
            semantic_score,
            experience_score,
            len(matched),
            len(missing)
        )

        decision = "SELECTED" if candidate_score >= 75 else "REJECTED"

        return {
            "Decision": decision,
            "Probability": probability,
            "Keyword Coverage": keyword_score,
            "Semantic Similarity": semantic_score,
            "Experience Match": f"{resume_exp}/{job_exp}",
            "Experience Score": experience_score,
            "Candidate Score": candidate_score,
            "Matched Skills": matched,
            "Missing Skills": missing,
            "Matched Count": len(matched),
            "Missing Count": len(missing)
        }

    result = predict_candidate(
        resume_text=resume_text,
        job_description=job_description
    )

    result["SN"] = idx
    result["Candidate Name"] = pdf_file.stem

    results_list.append(result)

print("\nFinished Processing All Candidates")

Processing Resume 1/8
AWS-Engineer-Resume
Processing Resume 2/8
Azure-Engineer-Resume
Processing Resume 3/8
Cloud-Engineer-Resume
Processing Resume 4/8
DevOps-Engineer-Resume
Processing Resume 5/8
DevSecOps-Engineer-Resume
Processing Resume 6/8
Infrastructure-Engineer-Resume
Processing Resume 7/8
Platform-Engineer-Resume
Processing Resume 8/8
Site-Reliability-Engineer-Resume

Finished Processing All Candidates


## SECTION 14 : Candidate Ranking

In [23]:
# ============================================================
# SECTION 14 : Candidate Ranking
# ============================================================

# Convert to DataFrame
ranking = pd.DataFrame(results_list)

# ------------------------------------------------------------
# Rename Columns for Professional Report
# ------------------------------------------------------------

ranking.rename(
    columns={
        "Probability": "Selection Probability"
    },
    inplace=True
)

# ------------------------------------------------------------
# Sort by Candidate Score
# ------------------------------------------------------------

ranking = ranking.sort_values(
    by="Candidate Score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Recreate Serial Number
# ------------------------------------------------------------

ranking["SN"] = range(1, len(ranking) + 1)

# ------------------------------------------------------------
# Columns to Display
# ------------------------------------------------------------

display_columns = [

    "SN",

    "Candidate Name",

    "Decision",

    "Candidate Score",

    "Selection Probability",

    "Keyword Coverage",

    "Semantic Similarity",

    "Experience Match",

    "Matched Count",

    "Missing Count",

    "Matched Skills",

    "Missing Skills"

]

# ------------------------------------------------------------
# Display Ranking
# ------------------------------------------------------------

display(

    ranking[display_columns]

    .style

    .background_gradient(

        subset=["Candidate Score"],

        cmap="Greens"

    )

    .format({

        "Candidate Score": "{:.2f}",

        "Selection Probability": "{:.2f}",

        "Keyword Coverage": "{:.2f}",

        "Semantic Similarity": "{:.3f}"

    })

)

print("\nRanking Generated Successfully")

,SN,Candidate Name,Decision,Candidate Score,Selection Probability,Keyword Coverage,Semantic Similarity,Experience Match,Matched Count,Missing Count,Matched Skills,Missing Skills
0,1,DevOps-Engineer-Resume,SELECTED,85.87,65.86,83.87,0.855,6/3,26,5,"['agile', 'automation', 'aws', 'azure', 'bash', 'cloud', 'cloudformation', 'cloudwatch', 'devops', 'docker', 'docker kubernetes', 'git', 'git github', 'github', 'grafana', 'helm', 'iam', 'jenkins', 'kubernetes', 'monitoring', 'networking', 'prometheus', 'python', 'security', 'terraform', 'vpc']","['ansible', 'cloud security', 'communication', 'gcp', 'linux']"
1,2,Infrastructure-Engineer-Resume,SELECTED,76.86,2.92,77.42,0.767,7/3,24,7,"['agile', 'ansible', 'automation', 'aws', 'azure', 'bash', 'cloud', 'cloudformation', 'docker', 'gcp', 'github', 'grafana', 'helm', 'iam', 'jenkins', 'kubernetes', 'linux', 'monitoring', 'networking', 'prometheus', 'python', 'security', 'terraform', 'vpc']","['cloud security', 'cloudwatch', 'communication', 'devops', 'docker kubernetes', 'git', 'git github']"
2,3,Cloud-Engineer-Resume,REJECTED,72.30,1.57,67.74,0.778,6/3,21,10,"['agile', 'ansible', 'automation', 'aws', 'azure', 'bash', 'cloud', 'cloudformation', 'cloudwatch', 'docker', 'docker kubernetes', 'gcp', 'github', 'iam', 'jenkins', 'kubernetes', 'networking', 'python', 'security', 'terraform', 'vpc']","['cloud security', 'communication', 'devops', 'git', 'git github', 'grafana', 'helm', 'linux', 'monitoring', 'prometheus']"
3,4,Site-Reliability-Engineer-Resume,REJECTED,66.92,25.86,61.29,0.666,7/3,19,12,"['agile', 'ansible', 'automation', 'aws', 'bash', 'cloud', 'docker', 'gcp', 'github', 'grafana', 'helm', 'kubernetes', 'linux', 'monitoring', 'networking', 'prometheus', 'python', 'security', 'terraform']","['azure', 'cloud security', 'cloudformation', 'cloudwatch', 'communication', 'devops', 'docker kubernetes', 'git', 'git github', 'iam', 'jenkins', 'vpc']"
4,5,AWS-Engineer-Resume,REJECTED,64.34,16.06,48.39,0.811,8/3,15,16,"['automation', 'aws', 'cloud', 'cloudformation', 'cloudwatch', 'devops', 'github', 'grafana', 'iam', 'jenkins', 'networking', 'python', 'security', 'terraform', 'vpc']","['agile', 'ansible', 'azure', 'bash', 'cloud security', 'communication', 'docker', 'docker kubernetes', 'gcp', 'git', 'git github', 'helm', 'kubernetes', 'linux', 'monitoring', 'prometheus']"
5,6,Platform-Engineer-Resume,REJECTED,57.67,9.29,45.16,0.654,7/3,14,17,"['aws', 'bash', 'cloud', 'gcp', 'github', 'grafana', 'helm', 'iam', 'kubernetes', 'networking', 'prometheus', 'python', 'security', 'terraform']","['agile', 'ansible', 'automation', 'azure', 'cloud security', 'cloudformation', 'cloudwatch', 'communication', 'devops', 'docker', 'docker kubernetes', 'git', 'git github', 'jenkins', 'linux', 'monitoring', 'vpc']"
6,7,DevSecOps-Engineer-Resume,REJECTED,56.62,11.44,41.94,0.669,7/3,13,18,"['automation', 'aws', 'bash', 'cloud', 'cloud security', 'cloudformation', 'github', 'helm', 'jenkins', 'kubernetes', 'python', 'security', 'terraform']","['agile', 'ansible', 'azure', 'cloudwatch', 'communication', 'devops', 'docker', 'docker kubernetes', 'gcp', 'git', 'git github', 'grafana', 'iam', 'linux', 'monitoring', 'networking', 'prometheus', 'vpc']"
7,8,Azure-Engineer-Resume,REJECTED,52.78,43.16,29.03,0.704,8/3,9,22,"['automation', 'azure', 'cloud', 'devops', 'github', 'grafana', 'networking', 'security', 'terraform']","['agile', 'ansible', 'aws', 'bash', 'cloud security', 'cloudformation', 'cloudwatch', 'communication', 'docker', 'docker kubernetes', 'gcp', 'git', 'git github', 'helm', 'iam', 'jenkins', 'kubernetes', 'linux', 'monitoring', 'prometheus', 'python', 'vpc']"



Ranking Generated Successfully


## Section 15 : Export CSV

In [24]:
from datetime import datetime

OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

csv_file = OUTPUT_DIR / (

    "candidate_ranking_"

    + datetime.now().strftime("%Y%m%d_%H%M%S")

    + ".csv"

)

ranking.to_csv(

    csv_file,

    index=False

)

print("CSV Saved Successfully")
print(csv_file)

CSV Saved Successfully
..\outputs\candidate_ranking_20260805_131013.csv


## SECTION 16 : Professional Report

In [25]:
print("\n")
print("=" * 90)
print(" FAIRHIRE PROFESSIONAL CANDIDATE REPORT ".center(90))
print("=" * 90)

for _, row in ranking.iterrows():

    print()

    print("-" * 90)

    print(f"SN                    : {row['SN']}")

    print(f"Candidate Name        : {row['Candidate Name']}")

    print(f"Decision              : {row['Decision']}")

    print(f"Candidate Score       : {row['Candidate Score']:.2f}%")

    print(f"Selection Probability : {row['Selection Probability']:.2f}%")

    print(f"Keyword Coverage      : {row['Keyword Coverage']:.2f}%")

    print(f"Semantic Similarity   : {row['Semantic Similarity']:.3f}")

    print(f"Experience Match      : {row['Experience Match']}")

    print(f"Matched Skills        : {row['Matched Count']}")

    print(f"Missing Skills        : {row['Missing Count']}")

    print()

    print("Matched Skill List")

    print("------------------")

    print(", ".join(row["Matched Skills"]))

    print()

    print("Missing Skill List")

    print("------------------")

    print(", ".join(row["Missing Skills"]))

    print("-" * 90)

print()

print("=" * 90)

print("Top Ranked Candidate")

print("=" * 90)

top = ranking.iloc[0]

print(f"Candidate : {top['Candidate Name']}")

print(f"Candidate Score : {top['Candidate Score']:.2f}%")

print(f"Decision : {top['Decision']}")



                          FAIRHIRE PROFESSIONAL CANDIDATE REPORT                          

------------------------------------------------------------------------------------------
SN                    : 1
Candidate Name        : DevOps-Engineer-Resume
Decision              : SELECTED
Candidate Score       : 85.87%
Selection Probability : 65.86%
Keyword Coverage      : 83.87%
Semantic Similarity   : 0.855
Experience Match      : 6/3
Matched Skills        : 26
Missing Skills        : 5

Matched Skill List
------------------
agile, automation, aws, azure, bash, cloud, cloudformation, cloudwatch, devops, docker, docker kubernetes, git, git github, github, grafana, helm, iam, jenkins, kubernetes, monitoring, networking, prometheus, python, security, terraform, vpc

Missing Skill List
------------------
ansible, cloud security, communication, gcp, linux
------------------------------------------------------------------------------------------

------------------------------------------